In [0]:
# 1. Define Widgets (Parameters passed from ADF)
dbutils.widgets.text("landed_path", "")
dbutils.widgets.text("format", "")
dbutils.widgets.text("read_options", "")
dbutils.widgets.text("bronze_table", "")

In [0]:
%run /Workspace/Repos/aslam123gayas@gmail.com/azurehumancapitalpwc/HC_Azureporject/generic_audit_notebook

In [0]:
# 2. Get values from Widgets
from datetime import datetime
from pyspark.sql.functions import current_timestamp, col

# ── Timing ────────────────────────────────────────────────────────────────────
run_start_ts = datetime.now()
LOAD_DATE    = run_start_ts.date()

# ── Widget Parameters ─────────────────────────────────────────────────────────
landed_path      = dbutils.widgets.get("landed_path")
fmt_raw          = dbutils.widgets.get("format")          # original (e.g. "tsv")
# Map non-standard format names to Spark-compatible equivalents
FORMAT_ALIASES = {
    "tsv":     "csv",   # tab-separated — use CSV reader with sep=\t
    "txt":     "csv",   # delimited text files
    "tab":     "csv",   # alternate tab-separated label
    "jsonl":   "json",  # JSON lines / newline-delimited JSON
    "ndjson":  "json",  # another NDJSON alias
}
file_format = FORMAT_ALIASES.get(fmt_raw.lower(), fmt_raw.lower())
read_options_str = dbutils.widgets.get("read_options")
bronze_table     = dbutils.widgets.get("bronze_table")    # e.g. "bronze.departments"

# ── Derived Source Metadata ───────────────────────────────────────────────────
source_name    = bronze_table.split(".")[-1]              # "departments"
full_table     = f"hcdatabrickspwc.{bronze_table}"        # fully qualified
ingestion_user = spark.sql("SELECT current_user()").collect()[0][0]
run_id         = f"run_{run_start_ts.strftime('%Y%m%d_%H%M%S')}_{source_name}"

# 3. Parse read_options string into a dictionary
options_dict = {}
if read_options_str and read_options_str.lower() != "null":
    for option in read_options_str.split(";"):
        if "=" in option:
            key, value = option.split("=", 1)
            key = key.strip()
            value = value.strip(' ')   # spaces only — tab/newline are valid delimiters
            # Unescape \t → tab, \n → newline etc. (handles both ADF literal strings and real chars)
            try:
                value = value.encode('raw_unicode_escape').decode('unicode_escape')
            except (UnicodeDecodeError, ValueError):
                pass
            options_dict[key] = value

try:
    # ── 4. Read from Landing ──────────────────────────────────────────────────
    print(f"[INFO] Reading: {landed_path}  |  format: {file_format}")
    df = spark.read.format(file_format).options(**options_dict).load(landed_path)
    records_read = df.count()
    print(f"[INFO] Records read: {records_read:,}")

    # ── 5. Add Audit Columns ──────────────────────────────────────────────────
    df_with_audit = (df
                     .withColumn("ingestion_timestamp", current_timestamp())
                     .withColumn("source_file", col("_metadata.file_path")))

    # ── 6. Write to Bronze Delta Table ────────────────────────────────────────
    print(f"[INFO] Writing to: {full_table}")
    (df_with_audit.write
     .format("delta")
     .mode("overwrite")
     .option("mergeSchema", "true")
     .saveAsTable(full_table))

    run_end_ts      = datetime.now()
    records_written = records_read    # full overwrite — all records written
    duration_ms     = int((run_end_ts - run_start_ts).total_seconds() * 1000)
    print(f"[INFO] Done. {records_written:,} records written | {duration_ms:,} ms")

    # ── 7. Audit Log — SUCCESS ────────────────────────────────────────────────
    log_audit_ingestion(
        pipeline_name        = "pl_landing_to_bronze",
        layer                = "bronze",
        source_name          = source_name,
        target_table         = full_table,
        source_type          = fmt_raw,
        bronze_table         = bronze_table,
        batch_id             = uuid.uuid4(),
        run_id               = run_id,
        trigger_type         = "scheduled",
        run_start_ts         = run_start_ts,
        run_end_ts           = run_end_ts,
        last_status          = "SUCCESS",
        records_read         = records_read,
        records_written      = records_written,
        error_count          = 0,
        file_checkpoint_path = landed_path,
        ingestion_user       = ingestion_user,
        last_load_date       = LOAD_DATE,
        notes                = f"ADF ForEach | source: {landed_path}"
    )
    print(f"[AUDIT] SUCCESS logged for {source_name} -> {full_table}")

except Exception as e:
    run_end_ts = datetime.now()
    error_msg  = str(e)
    print(f"[ERROR] Failed: {source_name} — {error_msg[:300]}")

    # ── 7b. Audit Log — FAILED ────────────────────────────────────────────────
    try:
        log_audit_ingestion(
            pipeline_name        = "pl_landing_to_bronze",
            layer                = "bronze",
            source_name          = source_name,
            target_table         = full_table,
            source_type          = fmt_raw,
            bronze_table         = bronze_table,
            run_id               = run_id,
            batch_id             = uuid.uuid4(),
            trigger_type         = "scheduled",
            run_start_ts         = run_start_ts,
            run_end_ts           = run_end_ts,
            last_status          = "FAILED",
            records_read         = 0,
            records_written      = 0,
            error_count          = 1,
            error_message        = error_msg[:2000],
            file_checkpoint_path = landed_path,
            ingestion_user       = ingestion_user,
            last_load_date       = LOAD_DATE,
            notes                = f"ADF ForEach | source: {landed_path}"
        )
        print(f"[AUDIT] FAILED logged for {source_name}")
    except Exception as audit_err:
        print(f"[WARN] Audit write failed: {audit_err}")

    raise  # Re-raise so ADF marks the ForEach activity as failed